In [ ]:
# ========== 安装依赖：OpenAI 兼容客户端、Pydantic、RSS、Gradio 等 ==========
# Colab 上通常需要；本地已装过时 -q 安静安装，一般无害
# Install required packages (needed on Colab; harmless if already installed locally)
!pip install -q openai pydantic python-dotenv requests feedparser gradio


In [ ]:
# ========== 环境密钥：Colab Secrets 或本地 .env ==========

# 标准库 os：读写环境变量（API Key、Pushover 凭据）
import os

# 自动检测：能否 import google.colab.userdata → 判断是否在 Colab
# Auto-detect if we are running on Google Colab or locally
try:
    # Colab 专用：从 Secrets 面板读密钥
    from google.colab import userdata
    # 标记：当前在 Colab 运行
    ON_COLAB = True
except ImportError:
    # 本地/普通 Jupyter：没有 google.colab
    ON_COLAB = False

if ON_COLAB:
    # 在 Colab 上：从 Secrets 面板中拉出（左侧边栏上的锁图标）
    # On Colab: pull from the Secrets panel (the lock icon on the left sidebar)
    # 确保这些确切的名称保存在那里：GROQ_API_KEY、OPENROUTER_API_KEY、PUSHOVER_USER、PUSHOVER_TOKEN
    # Make sure these exact names are saved there: GROQ_API_KEY, OPENROUTER_API_KEY, PUSHOVER_USER, PUSHOVER_TOKEN
    print("Running on Colab — loading keys from Colab Secrets...")
    # 逐个密钥名尝试加载；失败则提示去 Secrets 面板添加
    for key_name in ["GROQ_API_KEY", "OPENROUTER_API_KEY", "PUSHOVER_USER", "PUSHOVER_TOKEN"]:
        try:
            # userdata.get：从 Colab Secrets 取字符串写入 os.environ
            os.environ[key_name] = userdata.get(key_name)
            print(f"  ✅ {key_name} loaded from Colab Secrets")
        except Exception:
            print(f"  ❌ {key_name} not found in Colab Secrets — add it via the 🔑 panel")
else:
    # 在本地计算机上：从项目根目录中的 .env 文件加载
    # On local machine: load from the .env file in the project root
    from dotenv import load_dotenv
    # override=True：.env 覆盖已有同名环境变量
    load_dotenv(override=True)
    print("Running locally — loading keys from .env file...")
    # 检查四个关键密钥是否已在环境中（不打印密钥本身）
    for key_name in ["GROQ_API_KEY", "OPENROUTER_API_KEY", "PUSHOVER_USER", "PUSHOVER_TOKEN"]:
        status = "✅ Set" if os.environ.get(key_name) else "❌ Missing"
        print(f"  {key_name:30s} {status}")


# 第 8 周练习：带有信任验证的交易搜寻框架

## 练习目标（理念）

演示如何搭建 Ed Donner 第 8 周风格的**自主交易搜寻**流水线：从真实 RSS 抓优惠 → LLM 结构化抽取 → **信任校验** → 估价 → 达阈值则 Pushover 推送。

因为没有活跃的 OpenAI 计费账户，这里用第 1/2 周学过的 **OpenAI Wrapper Trick**：用 OpenAI 兼容客户端在 **Groq**、**Gemini**、**OpenRouter (Claude)** 之间轮询回退。

## 和第 8 周概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多代理虚拟公司 | Scanner / TrustVerification / Appraiser / Messaging |
| Planning 编排 | `run_deal_hunter_workflow` 串起扫描→安全→估价→告警 |
| 工具与外部世界 | RSS（feedparser）、Pushover HTTP API、Gradio UI |
| 新增安全代理 | **TrustVerificationAgent**：估价/推送前过滤骗局与破损品 |

## 怎么跑

1. 先跑安装与密钥检测格
2. 依次执行数据模型、基类 Agent、各专业代理、工作流与 Gradio
3. 需要至少一个 LLM 提供商密钥；推送还需 `PUSHOVER_USER` / `PUSHOVER_TOKEN`


In [ ]:
# ========== 导入：环境、HTTP、类型注解、Pydantic、OpenAI 兼容客户端 ==========

# 标准库 os：读 API Key 等环境变量
import os
# 标准库 json：解析 LLM 返回的 JSON、读写 seen_deals
import json
# requests：调用 Pushover HTTP API 发推送
import requests
# typing：List / Optional / Dict 标注消息与返回值
from typing import List, Optional, Dict
# pydantic：用 BaseModel 约束 Deal / Opportunity 字段
from pydantic import BaseModel, Field
# OpenAI 官方 SDK：也用来接 Groq / Gemini / OpenRouter 的兼容端点
from openai import OpenAI
# dotenv：从 .env 加载密钥（与上一格本地路径呼应）
from dotenv import load_dotenv

# 再次加载 .env（override），保证后续 Agent 构造时环境已就绪
load_dotenv(override=True)


In [ ]:
# ========== 1. 数据模型与 Schema（Pydantic）==========
# --------------------------------------------------------- #
# 1. 数据模型和模式
# 1. DATA MODELS & SCHEMAS
# --------------------------------------------------------- #

# Deal：一条结构化优惠（描述、成交价、链接、卖家信誉、成色）
class Deal(BaseModel):
    # 商品描述与关键规格
    product_description: str
    # 成交/标价（美元浮点）
    price: float
    # 购买链接
    url: str
    # 卖家或店铺信誉描述（可含星级，未知则为 Unknown）
    seller_reputation: str
    # 成色：New / Refurbished / Used / Open Box 等
    condition: str

# Opportunity：在 Deal 上叠加估价、折扣与信任校验结果
class Opportunity(BaseModel):
    # 原始优惠对象
    deal: Deal
    # LLM 估出的公允市价（Fair Market Value）
    estimate: float
    # 利润空间 ≈ estimate - price
    discount: float
    # 是否通过 TrustVerificationAgent
    is_verified: bool = False
    # 校验说明（模型返回的 APPROVE/REJECT 原文）
    verification_notes: str = ""


In [ ]:
# ========== 2. 基类 Agent：多提供商回退 + 统一 completion ==========
# --------------------------------------------------------- #
# 2. 代理人（虚拟公司）
# 2. THE AGENTS (The Virtual Company)
# --------------------------------------------------------- #
class Agent:
    def __init__(self):
        # 多提供商后备策略：Groq → Gemini → OpenRouter(Claude)
        # Multi-provider fallback strategy
        self.providers = [
            {
                "name": "Groq",
                # OpenAI 兼容客户端，指向 Groq 的 OpenAI-style base_url
                "client": OpenAI(api_key=os.getenv("GROQ_API_KEY", "missing"), base_url="https://api.groq.com/openai/v1"),
                "model": "llama-3.3-70b-versatile"
            },
            {
                "name": "Gemini",
                # Google Generative Language 的 OpenAI 兼容端点
                "client": OpenAI(api_key=os.getenv("GEMINI_API_KEY", "missing"), base_url="https://generativelanguage.googleapis.com/v1beta/openai/"),
                "model": "gemini-2.0-flash"
            },
            {
                "name": "OpenRouter (Claude)",
                # OpenRouter 统一网关；模型 id 指向 Claude 3.5 Sonnet
                "client": OpenAI(api_key=os.getenv("OPENROUTER_API_KEY", "missing"), base_url="https://openrouter.ai/api/v1"),
                "model": "anthropic/claude-3-5-sonnet"
            }
        ]
        
    def log(self, message: str):
        # 用类名做前缀，方便在 Gradio 日志里区分哪个代理在说话
        print(f"[{self.__class__.__name__}] {message}")
        
    def get_completion(self, messages: List[Dict], expect_json: bool = False) -> Optional[str]:
        """ Rotates through available keys until one works """
        # 依次尝试每个提供商，直到有一个成功返回
        for provider in self.providers:
            # 跳过占位 missing 或空 key 的客户端
            if provider["client"].api_key == "missing" or not provider["client"].api_key:
                continue
                
            try:
                # 组装 chat.completions 参数：模型名 + 消息列表
                kwargs = {
                    "model": provider["model"],
                    "messages": messages,
                }
                # 若需要 JSON，且不是挑剔的 OpenRouter，则强制 response_format=json_object
                # Add JSON object enforcement if requested and not on OpenRouter which can be finicky
                if expect_json and provider["name"] != "OpenRouter (Claude)":
                    kwargs["response_format"] = {"type": "json_object"}
                    
                # 真正发起聊天补全请求
                response = provider["client"].chat.completions.create(**kwargs)
                # 取第一条 choice 的文本并去首尾空白
                return response.choices[0].message.content.strip()
                
            except Exception as e:
                # 当前提供商失败：记日志，试下一个
                self.log(f"⚠ {provider['name']} failed: {e}. Trying next provider...")
                continue
                
        # 全部失败或无可用密钥
        self.log("❌ ERROR: All LLM API providers failed or missing keys.")
        return None


In [ ]:
# ========== ScannerAgent：RSS 抓取 → 去重 → LLM 抽成 Deal 列表 ==========

# feedparser：解析 RSS/Atom 优惠源
import feedparser
# re：用正则剥掉 HTML 标签、压空白
import re
# html：unescape 把 &amp; 等实体还原成字符
import html
# os：判断 seen_deals.json 是否存在
import os
# json：读写已见 URL 集合、解析 LLM JSON
import json

class ScannerAgent(Agent):
    # 系统提示：从杂乱网页文本里最多抽 5 条科技/电子优惠（JSON schema 约束）——字符串保持原文
    SYSTEM_PROMPT = """You extract up to 5 of the best tech/electronics deals from messy deal website text.
    Respond strictly in valid JSON format with a single key 'deals' which is a list of objects.
    Each object must have exactly these keys:
    - 'product_description' (string: product name and key specs)
    - 'price' (float: the deal price, or 0.0 if not clearly stated)
    - 'url' (string: the deal link)
    - 'seller_reputation' (string: seller or store name with star rating if known, otherwise 'Unknown')
    - 'condition' (string: 'New', 'Refurbished', 'Used', or 'Open Box')
    Only include items with a clear price. Skip coupons, gift cards, or services."""

    # 两个公开 RSS 源（URL 保持原样，勿改）
    FEED_URLS = [
        "https://slickdeals.net/newsearch.php?mode=frontpage&searcharea=deals&q=electronics&rss=1",
        "https://www.dealnews.com/c196/Electronics/?rss=1",
    ]
    # 本地文件：记住已处理过的 deal URL，避免重复推送
    SEEN_DEALS_FILE = "seen_deals.json"

    def __init__(self):
        # 初始化父类（providers 列表）
        super().__init__()
        # 从磁盘加载已见 URL 集合
        self.seen_urls = self._load_seen_deals()

    def _load_seen_deals(self) -> set:
        # 若文件存在则读 JSON 列表转成 set；损坏则空集合
        if os.path.exists(self.SEEN_DEALS_FILE):
            try:
                with open(self.SEEN_DEALS_FILE, "r") as f:
                    return set(json.load(f))
            except Exception:
                return set()
        return set()

    def _save_seen_deals(self):
        # 把 set 写成 JSON 列表落盘
        with open(self.SEEN_DEALS_FILE, "w") as f:
            json.dump(list(self.seen_urls), f)

    def _clean_html(self, raw: str) -> str:
        # 去掉标签 → 反转义实体 → 压缩空白
        raw = re.sub(r'<[^>]+>', ' ', raw)
        raw = html.unescape(raw)
        return re.sub(r'\s+', ' ', raw).strip()

    def _fetch_deals_text(self) -> str:
        # 汇总各 RSS 条目成给 LLM 看的纯文本行
        lines = []
        for url in self.FEED_URLS:
            self.log(f"Fetching feed: {url[:60]}...")
            try:
                # 解析 RSS
                feed = feedparser.parse(url)
                # 每个源最多取前 10 条
                for entry in feed.entries[:10]:
                    link = entry.get("link", "")
                    # 如果我们已经看到过这个 URL，请跳过！
                    # Skip if we have already seen this URL!
                    if link in self.seen_urls:
                        continue
                    # 清洗标题与摘要，摘要截断到 200 字符
                    title = self._clean_html(entry.get("title", ""))
                    summary = self._clean_html(entry.get("summary", ""))
                    lines.append(f"- {title}: {summary[:200]}. Link: {link}")
            except Exception as e:
                self.log(f"Feed fetch failed for {url[:40]}: {e}")
        return "\n".join(lines)

    def scan(self) -> list:
        # 对外入口：抓取 → 调 LLM → 返回 Deal 列表
        self.log("Fetching LIVE deals from real RSS feeds...")
        raw_text = self._fetch_deals_text()

        # 没有新条目就直接返回空列表
        if not raw_text.strip():
            self.log("No NEW deals found in the feeds right now.")
            return []

        self.log(f"Scraped {len(raw_text.splitlines())} new distinct lines. Sending to LLM...")
        # 组装 chat messages：system 约束格式，user 塞入截断后的原文
        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": f"Extract deals from this text:\n{raw_text[:3000]}\nRespond only with JSON."}
        ]

        # expect_json=True：尽量强制 JSON 对象响应
        response_text = self.get_completion(messages, expect_json=True)
        if not response_text: return []

        try:
            # 兼容模型包了一层 ```json ... ``` 的情况
            if response_text.startswith("```json"):
                response_text = response_text[7:-3]
            elif response_text.startswith("```"):
                response_text = response_text[3:-3]
            # 解析 JSON 并校验成 Deal 模型
            data = json.loads(response_text)
            deals = [Deal(**item) for item in data.get("deals", [])]
            
            # 将所有这些 URL 标记为已查看，以便我们不再处理它们
            # Mark all these URLs as seen so we never process them again
            for d in deals:
                self.seen_urls.add(d.url)
            self._save_seen_deals()
            
            self.log(f"Extracted {len(deals)} structured deals from live data.")
            return deals
        except Exception as e:
            self.log(f"Failed to parse JSON: {e}")
            return []


In [ ]:
# ========== TrustVerificationAgent：估价前的安全/信誉过滤 ==========

class TrustVerificationAgent(Agent):
    """ The Safety Agent! Filters out scams, broken items, or bad sellers """
    
    def verify(self, deal: Deal) -> tuple[bool, str]:
        # 记录正在校验的商品（描述截断，避免日志过长）
        self.log(f"Verifying trust signals for: {deal.product_description[:30]}...")
        
        # 用户提示：根据成色与卖家信誉给出 APPROVE/REJECT（规则字符串保持英文原文）
        prompt = f"""
        Evaluate this deal for safety, authenticity, and return risk.
        Product: {deal.product_description}
        Condition: {deal.condition}
        Seller Reputation: {deal.seller_reputation}
        
        RULES:
        1. If condition indicates 'Broken', 'Cracked', 'For Parts', or 'As-Is', REJECT.
        2. If condition is 'Refurbished' OR 'Used' AND seller reputation is poor (< 4.0 stars or 'Unknown'), REJECT.
        3. If seller reputation is terrible (< 3.0 stars) regardless of condition, REJECT.
        4. Otherwise, APPROVE.
        
        Respond with EXACTLY ONE WORD first: either 'APPROVE' or 'REJECT', followed by a one-sentence reason.
        """
        
        # 只发 user 角色消息；不强制 JSON
        response_text = self.get_completion([{"role": "user", "content": prompt}])
        if not response_text:
            # API 全挂时保守拒绝
            return False, "Could not verify due to API failure."
            
        # 以 APPROVE 开头（忽略大小写）视为通过
        is_approved = response_text.strip().upper().startswith("APPROVE")
        self.log(f"Verdict: {response_text.strip()}")
        return is_approved, response_text.strip()


In [ ]:
# ========== AppraiserAgent：让 LLM 直接估公允市价（USD）==========

class AppraiserAgent(Agent):
    """ Determines the market value of the item using the LLM directly """
    
    def estimate(self, details: str) -> float:
        # 进度提示：描述前 20 字符
        print(f"   - Requesting market value for {details[:20]}...")
        # 提示要求只回一个数字（prompt 字符串保持原文）
        prompt = f"""
        Estimate the Fair Market Value (in USD) for this item in new/like-new condition:
        {details}
        Respond WITH A SINGLE NUMBER only (e.g. 750). No dollar signs.
        """
        response_text = self.get_completion([{"role": "user", "content": prompt}])
        if not response_text: return 0.0
        
        try:
            # 清理响应以获得浮动
            # Clean up the response to get just the float
            import re
            # 从回复里抠出第一个整数或小数
            numbers = re.findall(r"\d+\.\d+|\d+", response_text)
            return float(numbers[0]) if numbers else 0.0
        except:
            # 解析失败则估值为 0
            return 0.0


In [ ]:
# ========== MessagingAgent：组装告警文案并通过 Pushover 推送 ==========

class MessagingAgent(Agent):
    """ Outputs alerts to the user via Pushover """
    def alert(self, opp: Opportunity):
        # 控制台也打印一份，方便无手机时调试
        print(f"\n📲 [SENDING PUSH NOTIFICATION...]")
        # 推送正文：利润、商品、成本/估值、信任结论、购买链接（文案保持英文）
        message = (
            f"DEAL ALERT! Profit Margin: ${opp.discount:.2f}\n"
            f"Item: {opp.deal.product_description}\n"
            f"Cost: ${opp.deal.price} | Real Value: ${opp.estimate}\n"
            f"Trust Check: PASS ({opp.verification_notes})\n"
            f"Buy Link: {opp.deal.url}"
        )
        print(message)
        
        # 从环境变量读 Pushover 用户与应用 token
        pushover_user = os.getenv("PUSHOVER_USER")
        pushover_token = os.getenv("PUSHOVER_TOKEN")

        if not pushover_user or not pushover_token:
            # 缺密钥则跳过真实 HTTP 调用
            print("⚠️ Pushover keys missing in .env. Skipping actual API call.")
            return

        # form 表单字段：user / token / message / sound
        payload = {
            "user": pushover_user,
            "token": pushover_token,
            "message": message,
            "sound": "cashregister",
        }
        
        try:
            # POST 到 Pushover messages API（URL 保持原样）
            response = requests.post("https://api.pushover.net/1/messages.json", data=payload)
            if response.status_code == 200:
                print("✅ Push notification sent successfully to your device!")
            else:
                print(f"❌ Failed to send push notification: {response.text}")
        except Exception as e:
            print(f"❌ Error sending push notification: {e}")


In [ ]:
# ========== 3. Planning 循环：扫描 → 信任 → 估价 → 选最优推送 ==========

# io.StringIO：把 print 重定向到内存缓冲，供 Gradio 展示
import io
# sys：临时替换/恢复 stdout
import sys
# gradio：后面 UI 格使用（本格先导入以保持原 import 顺序）
import gradio as gr
# threading：后台定时扫描线程
import threading
# time：sleep 间隔
import time

# --------------------------------------------------------- #
# 3. 规划循环（框架协调器）
# 3. THE PLANNING LOOP (Framework Orchestrator)
# --------------------------------------------------------- #
def run_deal_hunter_workflow():
    # 将所有打印语句捕获到字符串缓冲区，以便我们可以在 Gradio UI 中显示它们
    # Capture all print statements to a string buffer so we can show them in Gradio UI
    old_stdout = sys.stdout
    sys.stdout = capture_buf = io.StringIO()
    
    print("=== Booting Price Hunter AI (with Safety Failsafes) ===\n")
    
    # 四个角色：实习生扫描、安保校验、估价师、秘书推送
    intern = ScannerAgent()
    security = TrustVerificationAgent()
    appraiser = AppraiserAgent()
    secretary = MessagingAgent()
    
    # 从 RSS 抽结构化优惠
    deals = intern.scan()
    if not deals:
        print("No new deals found, or API failure occurred.")
        # 恢复 stdout 后返回已捕获日志
        sys.stdout = old_stdout
        return capture_buf.getvalue()
        
    opportunities = []
    
    # 逐条：先信任校验，再估价，正利润才进候选
    for count, deal in enumerate(deals):
        print(f"\n--- Processing Deal #{count+1} ---")
        print(f"Found: {deal.product_description}")
        
        is_safe, reason = security.verify(deal)
        if not is_safe:
            print(f"❌ SKIPPING: Failed trust check (Reason: {reason})")
            continue
            
        # 公允市价估计
        est_value = appraiser.estimate(deal.product_description)
        # 折扣/利润 = 估值 - 成交价
        discount = est_value - deal.price
        
        if discount > 0:
            print(f"✅ APPROVED & PRICED: True value ${est_value}. Profit margin: ${discount:.2f}")
            opp = Opportunity(deal=deal, estimate=est_value, discount=discount, is_verified=True, verification_notes=reason)
            opportunities.append(opp)
            
    print("\n=======================================================")
    print("Workflow Complete. Dispatching Alerts...")
    if opportunities:
        # 按利润从高到低排序，取最优
        opportunities.sort(key=lambda x: x.discount, reverse=True)
        best_deal = opportunities[0]
        if best_deal.discount > 20.0:  # Threshold for pushing directly to phone
            # 超过 $20 才真的推手机
            secretary.alert(best_deal)
        else:
            print(f"Best deal yields ${best_deal.discount:.2f}. Not worth buzzing the phone.")
    else:
        print("No safe, profitable deals found today.")
        
    # 务必恢复 stdout，避免影响后续单元格
    sys.stdout = old_stdout
    return capture_buf.getvalue()


In [ ]:
# ========== 4. Gradio UI + 后台 5 分钟定时扫描 ==========
# --------------------------------------------------------- #
# 4. GRADIO UI 和后台定时器执行器
# 4. GRADIO UI AND BACKGROUND TIMER EXECUTOR
# --------------------------------------------------------- #

# 全局日志字符串：手动/定时运行都会往前拼接
logs_history = ""

def manual_run():
    # 按钮触发：跑一轮工作流并更新 logs_history
    global logs_history
    try:
        new_logs = run_deal_hunter_workflow()
        logs_history = "\n" + "*"*40 + "\n[MANUAL RUN]\n" + str(new_logs) + logs_history
    except Exception as e:
        # 把完整 traceback 写进日志区，方便排错
        import traceback
        err_out = traceback.format_exc()
        logs_history = "\n" + "*"*40 + f"\n[MANUAL RUN ERROR]\n{err_out}\n" + logs_history
    return logs_history

def background_worker():
    # 守护线程：每 300 秒自动扫描一次
    global logs_history
    while True:
        # 等待 5 分钟
        # Wait 5 minutes
        time.sleep(300)
        try:
            new_logs = run_deal_hunter_workflow()
            logs_history = "\n" + "*"*40 + "\n[TIMER EXECUTED]\n" + new_logs + logs_history
            print("Background scan completed.")
        except Exception as e:
            print(f"Background scan failed: {e}")

# 启动后台线程（daemon = True，因此当笔记本关闭时它会停止）
# Start the background thread (daemon = True so it stops when notebook is closed)
thread = threading.Thread(target=background_worker, daemon=True)
thread.start()

# 构建用户界面
# Build the UI
with gr.Blocks() as demo:
    # 标题与说明（Markdown 字符串保持原英文 UI 文案）
    gr.Markdown("# 🤖 Autonomous Deal Hunter & Trust Verifier")
    gr.Markdown("This app automatically scrapes live RSS feeds every 5 minutes in the background, runs them through the Trust AI and Pricing AI, and pushes notifications directly to your phone via Pushover if a safe, profitable deal is found.")
    
    with gr.Row():
        # 手动立刻跑一轮
        run_btn = gr.Button("Force Manual Scan Now", variant="primary")
        
    # 自动刷新以显示最新后台日志的文本框
    # A textbox that auto-refreshes to show the latest background logs
    # 使用自动刷新的解决方法：通过 Javascript 定期单击不可见按钮
    # Using a workaround for auto-refresh: an invisible button clicked periodically via Javascript
    output_logs = gr.Textbox(label="Agent Activity Logs", lines=25, interactive=False)
    
    # 按钮点击 → manual_run → 刷新日志框
    run_btn.click(fn=manual_run, inputs=[], outputs=[output_logs])
    
    # 内置负载 every=X 在较新的 Gradio 中可用，我们在这里安全地使用它
    # The built-in load every=X is available in newer Gradio, we use it here safely
    def refresh_logs():
        # Timer 回调：只读全局日志
        return logs_history
        
    # 每 5 秒把最新 logs_history 刷到 UI
    timer = gr.Timer(5)
    timer.tick(fn=refresh_logs, inputs=[], outputs=[output_logs])

# 使用公共 URL 启动应用程序
# Launch the app with a public URL
demo.launch(share=True, debug=False)
